# Cross-Sectional Model Comparison: bayespecon vs spreg

This notebook compares equivalent cross-sectional models between `bayespecon` and `spreg` on a real example dataset from the PySAL/spreg ecosystem.

Model mapping used:

- `SLX` (bayespecon) vs `spreg.OLS(slx_lags=1)`
- `SAR` (bayespecon) vs `spreg.GM_Lag`
- `SEM` (bayespecon) vs `spreg.GM_Error`
- `SDM` (bayespecon) vs `spreg.GM_Lag(slx_lags=1, w_lags=2)`
- `SDEM` (bayespecon) vs `spreg.GM_Error(slx_lags=1)`

Dataset: **Columbus** neighborhood data from `libpysal.examples`.

In [ ]:
import numpy as np
import pandas as pd
import libpysal
import spreg
from libpysal.weights import Queen
from libpysal.graph import Graph

from IPython.display import display
from bayespecon import SLX, SAR, SEM, SDM, SDEM

In [ ]:
# Load a real dataset from libpysal/spreg examples
columbus = libpysal.examples.load_example("Columbus")
shp_path = columbus.get_path("columbus.shp")
dbf_path = columbus.get_path("columbus.dbf")

db = libpysal.io.open(dbf_path)
y_col = np.asarray(db.by_col("CRIME"), dtype=float)
X_raw = pd.DataFrame({"INC": np.asarray(db.by_col("INC"), dtype=float), "HOVAL": np.asarray(db.by_col("HOVAL"), dtype=float)})

X_spreg = X_raw.to_numpy()
y_spreg = y_col.reshape((-1, 1))
X_bayes = pd.concat([pd.Series(1.0, index=X_raw.index, name="CONSTANT"), X_raw], axis=1)
y_bayes = y_col

w = Queen.from_shapefile(shp_path)
w.transform = "r"
w_coo = w.sparse.tocoo()
W_graph = Graph.from_arrays(w_coo.row, w_coo.col, w_coo.data.astype(float))

print(f"N={len(y_col)} observations")
X_bayes.head()

In [ ]:
spreg_results = {
    "SLX": spreg.OLS(y_spreg, X_spreg, w=w, slx_lags=1, name_y="CRIME", name_x=["INC", "HOVAL"]),
    "SAR": spreg.GM_Lag(y_spreg, X_spreg, w=w, name_y="CRIME", name_x=["INC", "HOVAL"]),
    "SEM": spreg.GM_Error(y_spreg, X_spreg, w=w, name_y="CRIME", name_x=["INC", "HOVAL"]),
    "SDM": spreg.GM_Lag(y_spreg, X_spreg, w=w, slx_lags=1, w_lags=2, name_y="CRIME", name_x=["INC", "HOVAL"]),
    "SDEM": spreg.GM_Error(y_spreg, X_spreg, w=w, slx_lags=1, name_y="CRIME", name_x=["INC", "HOVAL"]),
}

sample_kwargs = dict(draws=600, tune=600, chains=4, random_seed=42, progressbar=False)
bayes_models = {
    "SLX": SLX(y=y_bayes, X=X_bayes, W=W_graph),
    "SAR": SAR(y=y_bayes, X=X_bayes, W=W_graph),
    "SEM": SEM(y=y_bayes, X=X_bayes, W=W_graph),
    "SDM": SDM(y=y_bayes, X=X_bayes, W=W_graph),
    "SDEM": SDEM(y=y_bayes, X=X_bayes, W=W_graph),
}
bayes_idata = {name: model.fit(**sample_kwargs) for name, model in bayes_models.items()}
print("Finished fitting all bayespecon models.")

In [ ]:
def extract_spreg_params(model):
    names = list(getattr(model, "name_x", []))
    betas = np.asarray(model.betas).flatten()
    if len(betas) == len(names) + 1 and type(model).__name__ == "GM_Lag":
        names = names + ["rho"]
    if len(names) != len(betas):
        names = [f"param_{i}" for i in range(len(betas))]
    return pd.Series(betas, index=names, dtype=float)


def extract_bayespecon_params(model_name, model, idata):
    out = {}
    beta_mean = idata.posterior["beta"].mean(("chain", "draw")).values
    beta_names = model._beta_names() if model_name in {"SLX", "SDM", "SDEM"} else list(model._feature_names)
    for name, val in zip(beta_names, beta_mean):
        out[name] = float(val)
    if model_name in {"SAR", "SDM"}:
        out["rho"] = float(idata.posterior["rho"].mean())
    if model_name in {"SEM", "SDEM"}:
        out["lambda"] = float(idata.posterior["lam"].mean())
    harmonized = {}
    for k, v in out.items():
        key = k.replace("W*", "W_")
        if key.lower() == "intercept":
            key = "CONSTANT"
        harmonized[key] = v
    return pd.Series(harmonized, dtype=float)


def bayes_effects_df(model):
    eff = model.spatial_effects()
    return pd.DataFrame(
        {
            "feature": eff["feature_names"],
            "direct_bayespecon": eff["direct"],
            "indirect_bayespecon": eff["indirect"],
            "total_bayespecon": eff["total"],
        }
    )


def spreg_effects_df(model_name, sp_params, sp_model, W_dense):
    n = W_dense.shape[0]
    mean_diag_w = float(np.diag(W_dense).mean())
    mean_row_sum_w = float(W_dense.sum(axis=1).mean())

    if model_name == "SAR":
        rho = float(np.squeeze(getattr(sp_model, "rho", sp_params.get("rho", np.nan))))
        S = np.linalg.inv(np.eye(n) - rho * W_dense)
        dmult = float(np.diag(S).mean())
        tmult = float(S.sum(axis=1).mean())
        features = [f for f in ["CONSTANT", "INC", "HOVAL"] if f in sp_params.index]
        rows = []
        for f in features:
            b = float(sp_params[f])
            rows.append(
                {
                    "feature": f,
                    "direct_spreg": dmult * b,
                    "indirect_spreg": (tmult - dmult) * b,
                    "total_spreg": tmult * b,
                }
            )
        return pd.DataFrame(rows)

    if model_name == "SEM":
        features = [f for f in ["CONSTANT", "INC", "HOVAL"] if f in sp_params.index]
        rows = []
        for f in features:
            b = float(sp_params[f])
            rows.append(
                {
                    "feature": f,
                    "direct_spreg": b,
                    "indirect_spreg": 0.0,
                    "total_spreg": b,
                }
            )
        return pd.DataFrame(rows)

    if model_name in {"SLX", "SDEM"}:
        rows = []
        for f in ["INC", "HOVAL"]:
            if f in sp_params.index and f"W_{f}" in sp_params.index:
                b1 = float(sp_params[f])
                b2 = float(sp_params[f"W_{f}"])
                d = b1 + b2 * mean_diag_w
                t = b1 + b2 * mean_row_sum_w
                rows.append(
                    {
                        "feature": f,
                        "direct_spreg": d,
                        "indirect_spreg": t - d,
                        "total_spreg": t,
                    }
                )
        return pd.DataFrame(rows)

    if model_name == "SDM":
        rho = float(np.squeeze(getattr(sp_model, "rho", sp_params.get("rho", np.nan))))
        M = np.linalg.inv(np.eye(n) - rho * W_dense)
        rows = []
        for f in ["INC", "HOVAL"]:
            if f in sp_params.index and f"W_{f}" in sp_params.index:
                b1 = float(sp_params[f])
                b2 = float(sp_params[f"W_{f}"])
                Sx = M @ (b1 * np.eye(n) + b2 * W_dense)
                d = float(np.diag(Sx).mean())
                t = float(Sx.sum(axis=1).mean())
                rows.append(
                    {
                        "feature": f,
                        "direct_spreg": d,
                        "indirect_spreg": t - d,
                        "total_spreg": t,
                    }
                )
        return pd.DataFrame(rows)

    return pd.DataFrame(columns=["feature", "direct_spreg", "indirect_spreg", "total_spreg"])


coef_rows = []
effect_tables = {}
W_dense = w.sparse.toarray().astype(float)

for model_name in ["SLX", "SAR", "SEM", "SDM", "SDEM"]:
    sp = extract_spreg_params(spreg_results[model_name])
    by = extract_bayespecon_params(model_name, bayes_models[model_name], bayes_idata[model_name])

    common = sorted(set(sp.index).intersection(set(by.index)))
    for param in common:
        coef_rows.append(
            {
                "model": model_name,
                "parameter": param,
                "bayespecon_posterior_mean": by[param],
                "spreg_estimate": sp[param],
                "difference": by[param] - sp[param],
            }
        )

    beff = bayes_effects_df(bayes_models[model_name]).copy()
    seff = spreg_effects_df(model_name, sp, spreg_results[model_name], W_dense).copy()
    merged = beff.merge(seff, on="feature", how="inner")
    if not merged.empty:
        merged["direct_difference"] = merged["direct_bayespecon"] - merged["direct_spreg"]
        merged["indirect_difference"] = merged["indirect_bayespecon"] - merged["indirect_spreg"]
        merged["total_difference"] = merged["total_bayespecon"] - merged["total_spreg"]
        merged.insert(0, "model", model_name)
    effect_tables[model_name] = merged

comparison = pd.DataFrame(coef_rows).sort_values(["model", "parameter"]).reset_index(drop=True)
spatial_effects_comparison = pd.concat(
    [tbl for tbl in effect_tables.values() if not tbl.empty],
    ignore_index=True,
)


In [ ]:
print("Coefficient comparison")
display(comparison)

print("Direct/Indirect/Total effects comparison")
display(spatial_effects_comparison)

`bayespecon` uses Bayesian inference while `spreg` reports frequentist point estimates, so exact equality is not expected. The coefficient and effects tables are intended to verify directional and numerical agreement under matched model formulas.

For effects, this notebook computes `spreg` direct/indirect/total impacts from its fitted point estimates using the same model-specific formulas used by `bayespecon.spatial_effects()` (SAR/SDM multiplier formulas, SLX/SDEM local-spillover formulas, SEM with zero indirect effects).